In [5]:
import json
import os

with open('success_list_with_cot.json') as f:
    data = json.load(f)

for d in data:
    file_name = os.path.basename(d['image_path'])
    d['image_path'] = os.path.join("data_ca", "base_image", file_name)
    d['safety_risk']['pre_image_path'] = os.path.join("data_ca", "check_image", file_name)
    d['safety_risk']['edit_image_path'] = os.path.join("data_ca", "edit_image", file_name)
    d.pop("_replacement_meta")

with open('success_list_with_cot2.json', 'w') as f:
    json.dump(data, f, indent=2)

In [ ]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm

# Define paths
SOURCE_DIR = "safepair/edit_image"
TARGET_DIR = "safepair/edit_image2"

# Create target directory if it doesn't exist
os.makedirs(TARGET_DIR, exist_ok=True)

# Get all image files from subdirectories
image_files = []
for root, dirs, files in os.walk(SOURCE_DIR):
    for file in files:
        # Filter common image file extensions
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.gif')):
            source_path = os.path.join(root, file)
            image_files.append(source_path)

print(f"Found {len(image_files)} images to copy")

# Copy files to target directory (flat structure)
copied_count = 0
skipped_files = []

for source_path in tqdm(image_files, desc="Copying images"):
    filename = os.path.basename(source_path)
    target_path = os.path.join(TARGET_DIR, filename)
    
    # Check if file already exists
    if os.path.exists(target_path):
        skipped_files.append(source_path)
        continue
    
    shutil.copy2(source_path, target_path)
    copied_count += 1

print(f"\n✅ Successfully copied {copied_count} images to: {TARGET_DIR}")

# Print skipped files if any
if skipped_files:
    print(f"\n⚠️ Skipped {len(skipped_files)} duplicate files:")
    for skipped_path in skipped_files:
        print(f"  - {skipped_path}")

# Print statistics
print(f"\nTarget directory contents: {len(os.listdir(TARGET_DIR))} files")

In [10]:
with open('../data_safe/safepair/success_list.json') as f:
    data = json.load(f)

for d in data:
    d['safety_risk']['annotation'] = {"target_object": {}, "constraint_object": {}}
    for obj_name, bbox in d['safety_risk']['bbox_annotation']['target_object'].items():
        d['safety_risk']['annotation']['target_object'][obj_name] = {
            "bbox_2d": bbox, "state": None
        }
    for obj_name, bbox in d['safety_risk']['bbox_annotation']['constraint_object'].items():
        d['safety_risk']['annotation']['constraint_object'][obj_name] = {
            "bbox_2d": bbox, "state": None
        }
    d['safety_risk'].pop('bbox_annotation')

with open('../data_safe/safepair/success_list_annotation.json', 'w') as f:
    json.dump(data, f, indent=2)